In [1]:
import os
import warnings
from glob import glob

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.rdFingerprintGenerator import (
    FingeprintGenerator64,
    GetAtomPairGenerator,
    GetMorganGenerator,
    GetRDKitFPGenerator,
    GetTopologicalTorsionGenerator,
)
from feature_engine.selection import DropConstantFeatures, DropDuplicateFeatures

# Settings the warnings to be ignored
warnings.filterwarnings("ignore")

RDLogger.DisableLog("rdApp.*")
IPythonConsole.drawOptions.addAtomIndices = True
IPythonConsole.molSize = 600, 600


morgan_fpgen = GetMorganGenerator(radius=3, fpSize=2048, includeChirality=True)
atompair_fpgen = GetAtomPairGenerator(fpSize=2048, includeChirality=True)
rdkit_fpgen = GetRDKitFPGenerator(fpSize=2048)
topological_torsion_fpgen = GetTopologicalTorsionGenerator(fpSize=2048)

In [2]:
df = pd.read_csv("data/BDE.csv")


def add_path(row: pd.Series):
    sdf_path = "-".join(row["M"].split("-")[:-1])
    row["path"] = f"data/sdf/M/{sdf_path}.sdf"
    return row


def process_row(row: pd.Series):
    if os.path.exists(row["path"]):
        mol = Chem.MolFromMolFile(row["path"], removeHs=False)
    else:
        return None
    if mol is None:
        return None
    row["smiles"] = Chem.MolToSmiles(mol)
    return row


df = df.apply(add_path, axis=1)

df = df.apply(process_row, axis=1).dropna()

In [3]:
fe_o = Chem.MolFromSmarts("[Fe]-[OH]")
fe_other = Chem.MolFromSmarts("[Fe]-[NH0,SH0,FH0,ClH0,BrH0,IH0]")
fe_o_break = AllChem.ReactionFromSmarts("[Fe:1]-[OH:2]>>[Fe:1].[OH:2]")
fe_other_break = AllChem.ReactionFromSmarts(
    "[Fe:1]-[NH0,SH0,FH0,ClH0,BrH0,IH0:2]>>[Fe:1].[NH0,SH0,FH0,ClH0,BrH0,IH0:2]"
)

In [4]:
def mol_break_bond(mol: Chem.Mol):
    if mol.HasSubstructMatch(fe_o):
        fe, other = fe_o_break.RunReactants((mol,))[0]
    elif mol.HasSubstructMatch(fe_other):
        fe, other = fe_other_break.RunReactants((mol,))[0]
    else:
        return None, None
    Chem.SanitizeMol(fe)
    Chem.SanitizeMol(other)
    return fe, other


def get_mol_fp(mol: Chem.Mol, fpgen: FingeprintGenerator64) -> np.ndarray:
    return fpgen.GetCountFingerprintAsNumPy(mol)


def gen_fp_row(row: pd.Series, fpgen: FingeprintGenerator64):
    mol = Chem.MolFromSmiles(row["smiles"])
    fe, other = mol_break_bond(mol)
    if fe is None or other is None:
        return None
    fp_mol = get_mol_fp(mol, fpgen)
    fp_fe = get_mol_fp(fe, fpgen)
    fp_other = get_mol_fp(other, fpgen)
    return np.concatenate([fp_mol, fp_fe, fp_other], dtype=np.int64)


dcf = DropConstantFeatures()
ddf = DropDuplicateFeatures()

In [5]:
morgan_fps = df.apply(gen_fp_row, axis=1, result_type="expand", fpgen=morgan_fpgen)
morgan_fps["bde"] = df["BDE"]
morgan_fps.dropna(inplace=True)
morgan_fps.reset_index(drop=True, inplace=True)
morgan_fps = dcf.fit_transform(morgan_fps)
morgan_fps = ddf.fit_transform(morgan_fps)

rdkit_fps = df.apply(gen_fp_row, axis=1, result_type="expand", fpgen=rdkit_fpgen)
rdkit_fps["bde"] = df["BDE"]
rdkit_fps.dropna(inplace=True)
rdkit_fps.reset_index(drop=True, inplace=True)
rdkit_fps = dcf.fit_transform(rdkit_fps)
rdkit_fps = ddf.fit_transform(rdkit_fps)

topo_torsion_fps = df.apply(
    gen_fp_row, axis=1, result_type="expand", fpgen=topological_torsion_fpgen
)
topo_torsion_fps["bde"] = df["BDE"]
topo_torsion_fps.dropna(inplace=True)
topo_torsion_fps.reset_index(drop=True, inplace=True)
topo_torsion_fps = dcf.fit_transform(topo_torsion_fps)
topo_torsion_fps = ddf.fit_transform(topo_torsion_fps)

atompair_fps = df.apply(gen_fp_row, axis=1, result_type="expand", fpgen=atompair_fpgen)
atompair_fps["bde"] = df["BDE"]
atompair_fps.dropna(inplace=True)
atompair_fps.reset_index(drop=True, inplace=True)
atompair_fps = dcf.fit_transform(atompair_fps)
atompair_fps = ddf.fit_transform(atompair_fps)

In [6]:
import tqdm
from sklearn.ensemble import (
    AdaBoostRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

random_seed = 42

In [7]:
param_grid = {
    "LR": {},
    "RF": {
        "n_estimators": [100, 200, 300, 400, 500, 600],
        "max_depth": [None, 5, 10, 15, 20],
    },
    "ET": {
        "n_estimators": [100, 200, 300, 400, 500, 600],
        "max_depth": [None, 5, 10, 15, 20],
    },
    "KNN": {"n_neighbors": [5, 10, 15], "leaf_size": [25, 30, 35]},
    "DT": {
        "max_depth": [
            None,
            10,
            20,
        ],
        "min_samples_split": [2, 4, 6],
    },
    "GBR": {
        "n_estimators": [100, 200, 300, 400, 500, 600],
        "max_depth": [5, 10, 15, 20],
    },
    "ABR": {
        "n_estimators": [100, 200, 300, 400, 500, 600],
    },
    "HGBR": {
        "max_depth": [5, 10, 15, 20],
        "min_samples_leaf": [5, 10, 15, 20],
    },
}
models = {
    "LR": LinearRegression(n_jobs=-1),
    "RF": RandomForestRegressor(n_jobs=-1, random_state=random_seed),
    "ET": ExtraTreesRegressor(n_jobs=-1, random_state=random_seed),
    "KNN": KNeighborsRegressor(n_jobs=-1),
    "DT": DecisionTreeRegressor(random_state=random_seed),
    "GBR": GradientBoostingRegressor(random_state=random_seed),
    "ABR": AdaBoostRegressor(random_state=random_seed),
    "HGBR": HistGradientBoostingRegressor(random_state=random_seed),
}

In [8]:
train_index, test_index = train_test_split(
    morgan_fps.index, train_size=0.7, random_state=random_seed, shuffle=True
)
train_val, test_val = (
    morgan_fps["bde"].iloc[train_index],
    morgan_fps["bde"].iloc[test_index],
)

descs = {
    "morgan": morgan_fps,
    "rdkit": rdkit_fps,
    "topo": topo_torsion_fps,
    "atompair": atompair_fps,
}

In [ ]:
best_params = {}
kfold = KFold(n_splits=10, shuffle=True, random_state=random_seed)
with tqdm.tqdm(total=len(models) * len(descs)) as pbar:
    for model_name, model in models.items():
        for desc_name, desc in descs.items():
            train_desc = desc.drop("bde", axis=1)
            train_val = desc["bde"]
            GS = GridSearchCV(
                model, param_grid[model_name], cv=kfold, n_jobs=-1, scoring="r2"
            )
            GS.fit(train_desc, train_val)
            best_param = model.get_params()
            best_param.update(GS.best_params_)
            best_score = GS.best_score_

            best_params[(model_name, desc_name)] = best_param
            pbar.set_postfix_str(
                s="Model: {:4s}, Desc: {:8s}, Best Socre: {:4f}, Best Param: {}".format(
                    model_name, desc_name, best_score, best_param
                )
            )
            pbar.update()

In [10]:
np.save(os.path.join("result", "hyperparameters_opt.npy"), best_params)

In [ ]:
from scipy.stats import pearsonr
import re


performance_result = []
kfold = KFold(n_splits=10, shuffle=True, random_state=random_seed)
for model_name, desc_name in best_params.keys():
    desc = descs[desc_name]
    train_val = desc["bde"]
    test_val = train_val.iloc[test_index]

    desc = desc.drop("bde", axis=1)
    model = models[model_name]
    model.set_params(**best_params[(model_name, desc_name)])
    train_desc = desc.iloc[train_index]
    train_val = train_val.iloc[train_index]
    test_desc = desc.iloc[test_index]
    all_test_p, all_test_y = [], []
    for train_idx, test_idx in kfold.split(train_desc):
        train_x, test_x = train_desc.iloc[train_idx], train_desc.iloc[test_idx]
        train_y, test_y = (
            train_val.iloc[train_idx],
            train_val.iloc[test_idx],
        )
        model.fit(train_x, train_y)
        test_p = model.predict(test_desc)
        all_test_p.append(test_p)
        all_test_y.append(test_val)
    all_test_p = np.concatenate(all_test_p)
    all_test_y = np.concatenate(all_test_y)
    performance_result.append(
        pd.DataFrame(
            {
                "$yield_{observed}$": all_test_y,
                "$yield_{pred}$": all_test_p,
                "model": model_name,
                "desc": desc_name,
                "r2": r2_score(all_test_y, all_test_p),
                "mae": mean_absolute_error(all_test_y, all_test_p),
                "pearson_r": pearsonr(all_test_y, all_test_p)[0],
                "rmse": root_mean_squared_error(all_test_y, all_test_p),
            }
        )
    )
    print(
        f"Model: {model_name:10s}, Desc: {desc_name:10s}, R2: {r2_score(all_test_y, all_test_p):.4f}, "
        f"PearsonR: {pearsonr(all_test_y, all_test_p)[0]:.4f}, MAE: {mean_absolute_error(all_test_y, all_test_p):.4f}, "
        f"RMSE: {root_mean_squared_error(all_test_y, all_test_p):.4f}"
    )

In [12]:
performance_result_df = pd.concat(performance_result)
performance_result_df.to_csv(os.path.join("result", "performance_result.csv"))

In [ ]:
performance_result_df

In [15]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
def annotate(data, **kws):
    ax = plt.gca()
    ax.text(0.05, 0.9, f"r2 = {data['r2'].iloc[0]:5.3f}", transform=ax.transAxes)
    ax.text(
        0.05,
        0.85,
        f"pearson r = {data['pearson_r'].iloc[0]:.3f}",
        transform=ax.transAxes,
    )
    ax.text(0.05, 0.8, f"mae = {data['mae'].iloc[0]:.3f}", transform=ax.transAxes)
    ax.text(0.05, 0.75, f"rmse = {data['rmse'].iloc[0]:.3f}", transform=ax.transAxes)
    ax.plot(np.linspace(0, 200, 200), np.linspace(0, 200, 200), color="blue")


g = sns.FacetGrid(
    performance_result_df, col="model", row="desc", xlim=(0, 200), ylim=(0, 200)
)
g.map_dataframe(
    sns.scatterplot, "$yield_{observed}$", "$yield_{pred}$", color="lightgreen"
)
g.map_dataframe(annotate)